# NuclearOps: efficient trace storage
Enable `nuclear.save_trace_efficient = True` in your userscript settings before running. The default is False. This option takes precedence over `no_trace` and `save_smartly`: full analysis-input traces are saved, not sparse traces. Live result plotting remains available.

Each measurement folder contains `results.json` (typed result table), `measurement.json` (column metadata), and `traces/*.npy` with matching analysis-settings JSON files. Move/copy the **whole folder** together. The `trace` column contains relative paths; simultaneous result rows share one path. Unreferenced files can remain after interrupted or repeated acquisitions. New-format acquisition resume is not supported yet.

This reduces historical trace retention, not the working memory used to analyse one acquisition. It saves the trace supplied to NuclearOps analysis, not original TimeTagger timestamps. Other array-valued observations are unchanged. Standard loading/plots below need numpy, pandas and matplotlib; SSR/CRC reanalysis needs the Qudi environment.


In [ ]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

measurement = Path(r"F:/Data/NuclearOps/REPLACE_WITH_MEASUREMENT_FOLDER")
metadata = json.loads((measurement / "measurement.json").read_text())
assert metadata["format"] == "nuclearops-efficient-traces" and metadata["version"] == 1
results = pd.read_json(measurement / "results.json", orient="table")
display(results.head())
print("Parameters:", metadata["parameter_names"])


## Plot saved results
Choose a swept parameter for `x_column`, or leave it as None to plot measurement rows.

In [ ]:
x_column = None  # e.g. "smiq_freq"
y_column = next((c for c in results if c.startswith("result_")), "average_counts")
results.plot(x=x_column, y=y_column, marker=".", linewidth=0.7)
plt.show()


## Load one full trace
Memory mapping keeps the full file off the Python heap. Preview only a bounded number of points; identical references represent a shared acquisition.

In [ ]:
references = results["trace"].dropna().drop_duplicates().tolist()
reference = references[0]  # choose another acquisition here
trace_path = measurement / reference
raw = np.load(trace_path, mmap_mode="r", allow_pickle=False)
settings = json.loads(trace_path.with_suffix(".json").read_text())
display(results.loc[results["trace"] == reference])
print("Trace shape/dtype:", raw.shape, raw.dtype)
print("Analysis settings:", settings)
preview = raw[:10000]
plt.plot(preview)
plt.xlabel("Sample index (first 10,000 samples)")
plt.ylabel("Counts")
plt.show()


## Analyse counts without loading the full trace
This computes the overall mean and histogram in chunks. These are raw-count statistics, not postselected spin probabilities.

In [ ]:
chunk_size = 100_000
count = 0
count_sum = 0.0
histogram = {}
for start in range(0, raw.size, chunk_size):
    chunk = raw[start:start + chunk_size]
    count += chunk.size
    count_sum += chunk.sum(dtype=np.float64)
    values, frequencies = np.unique(chunk, return_counts=True)
    for value, frequency in zip(values, frequencies):
        histogram[int(value)] = histogram.get(int(value), 0) + int(frequency)
print("Mean counts:", count_sum / count if count else np.nan)
plt.bar(list(histogram), list(histogram.values()))
plt.xlabel("Counts per sample")
plt.ylabel("Frequency")
plt.show()


## Reanalyse SSR / CRC with your own thresholds
Run the following analysis cells in your **Qudi Python environment**, with this repository importable. They call the same `qudi.logic.Analysis.Trace.analyze()` implementation as NuclearOps, preserving its binning, postselection, consecutive-readout and averaging behaviour instead of approximating it with a count histogram.

First inspect the saved sequence below. Step numbers are zero-based. The file records `init` / `result` roles, not names such as SSR or CRC: identify the appropriate steps from your userscript. An `init` step is a postselection gate; a `result` step contributes a readout result.

For a single-memory step, Qudi compares its count with the threshold: operator `>` selects high counts and `<` selects low counts (digital class 0). Reliability additionally requires `abs(count - threshold) > exclusion`. With exclusion 0, counts exactly at the threshold are rejected. Ordinary postselection requires all init steps to select class 0 and all steps to be reliable. Consecutive mode follows Qudi's separate logic.

For **multiple memories**, Qudi chooses the minimum/maximum memory count and checks the gap between the best two; a scalar threshold is not used. Adjust `exclusion` for confidence in these SSR steps. Thresholds apply to Qudi's rebinned counts, not necessarily an individual acquisition sample.


In [ ]:
from copy import deepcopy

sequence_columns = ["role", "operator", "threshold", "repetitions", "exclusion", "memories"]
sequence_table = pd.DataFrame(
    [step[:6] for step in settings["analyze_sequence"]], columns=sequence_columns
)
sequence_table.index.name = "step"
display(sequence_table)
print("Mode:", settings["analyze_type"], "Binning:", settings["binning_factor"])


### Enter thresholds here
Edit `step_overrides`, then rerun this cell and the cells below. Empty overrides reproduce the saved settings. The commented examples illustrate syntax only: substitute the step numbers and thresholds appropriate for your measurement. You may override threshold, operator and exclusion independently; the saved sequence structure stays unchanged.


In [ ]:
step_overrides = {
    # 0: {"threshold": 5, "operator": ">", "exclusion": 0},  # e.g. CRC init step
    # 1: {"threshold": 12, "operator": ">", "exclusion": 2}, # e.g. single-memory SSR
    # 2: {"exclusion": 3},  # e.g. multi-memory SSR: minimum count gap
}


def settings_with_thresholds(saved_settings, overrides):
    configured = deepcopy(saved_settings)
    sequence = configured["analyze_sequence"]
    fields = {"threshold": 2, "operator": 1, "exclusion": 4}
    for index, changes in overrides.items():
        if not isinstance(index, int) or not 0 <= index < len(sequence):
            raise ValueError(f"Unknown step index: {index}")
        for field, value in changes.items():
            if field not in fields:
                raise ValueError(f"Unknown override: {field}; choose {list(fields)}")
            if field == "operator":
                if value not in ("<", ">"):
                    raise ValueError("operator must be '<' or '>'")
            else:
                if isinstance(value, bool) or not isinstance(value, (int, np.integer)):
                    raise ValueError(f"{field} must be an integer count")
                value = int(value)
                if field == "exclusion" and value < 0:
                    raise ValueError("exclusion must be nonnegative")
                if field == "threshold" and sequence[index][5] != 1:
                    raise ValueError(f"Step {index} uses multiple memories: set exclusion instead of threshold")
            sequence[index][fields[field]] = value
    return configured


analysis_settings = settings_with_thresholds(settings, step_overrides)
display(pd.DataFrame([s[:6] for s in analysis_settings["analyze_sequence"]],
                     columns=sequence_columns).rename_axis("step"))


### Compare saved-settings analysis with your thresholds
Each call processes one trace and releases its analysis object afterward. The underlying Qudi analysis still allocates temporary tables for that trace. `sm` identifies the simultaneous measurement, `step` the result step and (when not averaged) `result_num` the classified state. `events` reports Qudi's accepted-event count. No measurement files are changed.


In [ ]:
from qudi.logic.Analysis import Trace


def analyse_one_trace(array, configured_settings):
    analysis = Trace(trace=array, **configured_settings)
    if analysis.number_of_runs_rebinned < 1:
        raise ValueError("Trace is too short for the saved sequence and binning factor")
    return analysis.analyze().df.copy()


baseline = analyse_one_trace(raw, settings)
reanalysed = analyse_one_trace(raw, analysis_settings)
keys = [c for c in ("sm", "step", "result_num") if c in baseline and c in reanalysed]
comparison = baseline.merge(reanalysed, on=keys, how="outer", suffixes=("_saved_settings", "_new"))
comparison["result_change"] = comparison["result_new"] - comparison["result_saved_settings"]
display(comparison)
comparison.plot.bar(x=keys[-1], y=["result_saved_settings", "result_new"])
plt.ylabel("Qudi result")
plt.show()


### Optional: apply these thresholds to the whole measurement
Set `analyse_all = True` to load each unique trace once, using its own saved settings plus your overrides. Simultaneous result rows share a trace; `sm` maps outputs to those rows in acquisition order. A mismatched row count stops the operation instead of guessing. Check that step numbers mean the same thing across your measurement before applying overrides globally.

The output remains separate from the original results. Optional export writes new files with your threshold configuration; it does not overwrite acquisition data.


In [ ]:
analyse_all = False
export_reanalysis = False

if analyse_all:
    outputs = []
    for trace_reference in references:
        path = measurement / trace_reference
        per_trace_settings = json.loads(path.with_suffix(".json").read_text())
        configured = settings_with_thresholds(per_trace_settings, step_overrides)
        rows = results.loc[results["trace"] == trace_reference]
        n_sm = configured["number_of_simultaneous_measurements"]
        if len(rows) != n_sm:
            raise ValueError(f"{trace_reference}: expected {n_sm} result rows, found {len(rows)}")
        array = np.load(path, mmap_mode="r", allow_pickle=False)
        output = analyse_one_trace(array, configured)
        del array
        output["trace"] = trace_reference
        # Keep acquisition parameters alongside each reanalysed result.
        for parameter in metadata["parameter_names"]:
            output[parameter] = output["sm"].map(dict(enumerate(rows[parameter].tolist())))
        outputs.append(output)
    all_reanalysed = pd.concat(outputs, ignore_index=True)
    display(all_reanalysed.head())
    if export_reanalysis:
        from datetime import datetime
        stamp = datetime.now().strftime("%Y%m%d_%H%M%S_%f")
        destination = measurement / f"reanalysis_{stamp}.csv"
        all_reanalysed.to_csv(destination, index=False)
        destination.with_suffix(".json").write_text(json.dumps({
            "step_overrides": step_overrides,
            "analysis_engine": "qudi.logic.Analysis.Trace.analyze",
        }, indent=2))
        print("Exported:", destination)
